# Module 07: Insets and Zoom Windows

The previous module covered multi-panel layouts, where separate plots sit side by side in a grid. This module covers a different technique: embedding a smaller axes directly inside a larger one. This is the right tool when a specific region of a plot contains detail that deserves closer inspection, but removing it from the full-range view would cost the reader important context.

## Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

## 1. Placing an inset axis with `inset_axes`

`inset_axes` from `mpl_toolkits.axes_grid1.inset_locator` creates a new axes object that lives inside an existing one. You specify its size as a percentage of the parent axes and anchor it to one of the corners. The inset is a fully independent axes: it has its own limits, ticks, and content, and you interact with it exactly like any other `ax` object.

In [ ]:
# --- Simulate tensile test data ---

def tensile_curve(elongation, E, sigma_y, n, elongation_break):
    """Simple Ramberg-Osgood-like stress-strain curve, clipped at break."""
    stress = E * elongation / (1 + (E * elongation / sigma_y)**n)**(1/n)
    stress[elongation > elongation_break] = np.nan
    return stress

# Polymer A — ductile, breaks at ~50%
elong_A = np.linspace(0, 0.52, 500)
stress_A = tensile_curve(elong_A, E=500, sigma_y=25, n=5, elongation_break=0.50)

# Polymer B — ductile (stiffer), breaks at ~45%
elong_B = np.linspace(0, 0.47, 500)
stress_B = tensile_curve(elong_B, E=800, sigma_y=35, n=4, elongation_break=0.45)

# Polymer C — rigid/brittle, breaks at ~1%
elong_C = np.linspace(0, 0.012, 500)
stress_C = tensile_curve(elong_C, E=5000, sigma_y=80, n=12, elongation_break=0.010)

# --- Main plot ---
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(elong_A * 100, stress_A, color="#2196F3", linewidth=2,   label="Polymer A (ductile)")
ax.plot(elong_B * 100, stress_B, color="#4CAF50", linewidth=2,   label="Polymer B (ductile)")
ax.plot(elong_C * 100, stress_C, color="#F44336", linewidth=2,   label="Polymer C (rigid)")

ax.set_xlabel("Elongation (%)", fontsize=12)
ax.set_ylabel("Tensile Stress (MPa)", fontsize=12)
ax.set_title("Tensile Test — Three Polymers", fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(-1, 55)
ax.set_ylim(-2, 90)

# Annotate that C is barely visible at this scale
ax.annotate(
    "Polymer C breaks at ~1%\n→ see inset",
    xy=(1.0, 50), xytext=(8, 55),
    arrowprops=dict(arrowstyle="->", color="#F44336"),
    color="#F44336", fontsize=9
)

# --- Inset: zoom into 0–2% elongation ---
ax_inset = inset_axes(ax, width="38%", height="42%", loc="upper right")

ax_inset.plot(elong_A * 100, stress_A, color="#2196F3", linewidth=1.5)
ax_inset.plot(elong_B * 100, stress_B, color="#4CAF50", linewidth=1.5)
ax_inset.plot(elong_C * 100, stress_C, color="#F44336", linewidth=1.5)

ax_inset.set_xlim(0, 2)
ax_inset.set_ylim(0, 90)
ax_inset.set_xlabel("Elongation (%)", fontsize=8)
ax_inset.set_ylabel("Stress (MPa)", fontsize=8)
ax_inset.set_title("Zoom: 0–2% region", fontsize=8)
ax_inset.tick_params(labelsize=7)
plt.show()

The inset sits inside the main axes and zooms in on the **0–2 % elongation region**. `ax_inset` is an ordinary axes object: `set_xlim`, `set_ylim`, and every other axes method work exactly as they do on the parent. The `loc` parameter accepts the same location strings as legend placement.

Polymer C is far too rigid to be visible at the full scale (it fractures at ~1 % elongation), so the inset is the natural way to show all three materials in one figure without sacrificing detail.

## 2. Zoom insets with `mark_inset`

A zoom inset is a specific use of an inset where the inset shows a magnified region of the main plot, and connector lines link the zoomed region to its origin. `mark_inset` draws those connectors automatically once you have both axes set up. NMR spectroscopy is a natural fit for this technique: the full spectrum from 0 to 10 ppm tells you what chemical environments are present, but the fine structure of a single peak, a doublet, triplet, or quartet, is what reveals the neighboring hydrogens. The full spectrum and the zoomed peak need to be visible at the same time.

In [ ]:
def gaussian_peak(x, center, height, width):
    """Return a Gaussian peak evaluated over array x."""
    return height * np.exp(-((x - center) ** 2) / (2 * width ** 2))

# Build a synthetic 1H NMR spectrum over 0 to 10 ppm
ppm = np.linspace(0, 10, 8000)
spectrum = np.zeros_like(ppm)

# Singlet near 7.3 ppm (aromatic region)
spectrum += gaussian_peak(ppm, center=7.30, height=1.0, width=0.025)

# Doublet near 4.1 ppm (OCH2 adjacent to one H)
spectrum += gaussian_peak(ppm, center=4.08, height=0.65, width=0.018)
spectrum += gaussian_peak(ppm, center=4.14, height=0.65, width=0.018)

# Triplet near 2.3 ppm (CH2 adjacent to two equivalent H): 1:2:1 ratio
triplet_spacing = 0.03   # coupling constant in ppm units
triplet_center = 2.31
spectrum += gaussian_peak(ppm, center=triplet_center - triplet_spacing, height=0.40, width=0.012)
spectrum += gaussian_peak(ppm, center=triplet_center,                   height=0.80, width=0.012)
spectrum += gaussian_peak(ppm, center=triplet_center + triplet_spacing, height=0.40, width=0.012)

# Singlet near 1.2 ppm (methyl group)
spectrum += gaussian_peak(ppm, center=1.22, height=0.90, width=0.022)

# Small broad residual solvent peak near 3.3 ppm
spectrum += gaussian_peak(ppm, center=3.31, height=0.15, width=0.040)

# Add a very small noise floor to simulate a real spectrum
rng = np.random.default_rng(seed=42)
spectrum += rng.normal(0, 0.003, size=ppm.shape)

The spectrum is built by summing individual Gaussian functions, each representing one peak or one line of a multiplet. The triplet at 2.31 ppm has three components at equal spacing with heights in the 1:2:1 ratio that first-order coupling theory predicts. Try adjusting `triplet_spacing` and re-running to see how a larger coupling constant spreads the lines apart.

In [ ]:
fig, ax_nmr = plt.subplots(figsize=(10, 4))

ax_nmr.plot(ppm, spectrum, color="black", linewidth=0.9)

# NMR convention: chemical shift axis runs right to left
ax_nmr.set_xlim(10, 0)
ax_nmr.set_ylim(-0.05, 1.15)
ax_nmr.set_xlabel("Chemical shift (ppm)")
ax_nmr.set_ylabel("Intensity (a.u.)")
ax_nmr.set_title("Synthetic $^1$H NMR spectrum")

# Place the inset in the upper left corner (which appears far from the crowded aromatic region)
ax_zoom = inset_axes(
    ax_nmr,
    width="35%",
    height="55%",
    loc="upper left"
)

ax_zoom.plot(ppm, spectrum, color="black", linewidth=1.0)

# Zoom into the triplet region
zoom_center = triplet_center
zoom_half_width = 0.08
ax_zoom.set_xlim(zoom_center + zoom_half_width, zoom_center - zoom_half_width)  # keep right-to-left convention
ax_zoom.set_ylim(-0.02, 0.95)
ax_zoom.set_xlabel("ppm", fontsize=8)
ax_zoom.tick_params(labelsize=7)

# mark_inset draws the rectangle on the main axes and the connecting lines
# loc1 and loc2 specify which corners of the inset box connect to the main axes rectangle
# 1=lower left, 2=lower right, 3=upper right, 4=upper left (following Matplotlib corner numbering)
mark_inset(
    ax_nmr, ax_zoom,
    loc1=3, loc2=4,           # connect upper corners of inset to the highlighted region
    fc="lightyellow",         # fill colour of the rectangle on the main plot
    ec="gray",                # edge colour of the rectangle and connector lines
    linestyle="--",
    linewidth=0.8
)

plt.show()

`mark_inset` takes the parent axes and the inset axes, then draws a rectangle on the parent that corresponds to the inset's axis limits, and connects two corners of that rectangle to two corners of the inset box with lines. The `loc1` and `loc2` arguments choose which pair of corners gets connected; changing them avoids crossing lines when the inset is positioned on the opposite side of the highlighted region. The 1:2:1 height ratio of the triplet is now clearly visible in the inset, which would have been impossible to read at the scale of the full spectrum.

## 3. Controlling the inset independently

Because the inset is its own axes object, you have full independent control over its tick density, line styles, and axis limits. This matters when the zoomed-in region needs a different visual treatment from the main plot: finer tick marks to read off exact values, thicker lines so small peaks remain visible, or a different grid.

In [ ]:
fig, ax_main = plt.subplots(figsize=(9, 4))

ax_main.plot(ppm, spectrum, color="black", linewidth=0.8)
ax_main.set_xlim(10, 0)
ax_main.set_ylim(-0.05, 1.15)
ax_main.set_xlabel("Chemical shift (ppm)")
ax_main.set_ylabel("Intensity (a.u.)")
ax_main.set_title("Inset with customised tick density and line weight")

ax_detail = inset_axes(
    ax_main,
    width="38%",
    height="58%",
    loc="upper left"
)

# Plot the inset with a heavier line so fine peaks stand out
ax_detail.plot(ppm, spectrum, color="black", linewidth=1.8)

# Zoom in on the triplet with tighter limits
ax_detail.set_xlim(triplet_center + 0.07, triplet_center - 0.07)
ax_detail.set_ylim(-0.02, 0.92)

# Set explicit tick positions for the x-axis so they fall on the peak lines
ax_detail.set_xticks([
    triplet_center - triplet_spacing,
    triplet_center,
    triplet_center + triplet_spacing
])
ax_detail.xaxis.set_tick_params(labelsize=7)

# Add a light horizontal grid only to the inset to help read intensity ratios
ax_detail.yaxis.set_tick_params(labelsize=7)
ax_detail.set_yticks([0, 0.4, 0.8])
ax_detail.yaxis.tick_right()
ax_detail.patch.set_alpha(0.7)
ax_detail.grid(axis="y", linestyle=":", linewidth=0.6, color="gray", alpha=0.7)

mark_inset(
    ax_main, ax_detail,
    loc1=3, loc2=4,
    fc="lightyellow",
    ec="gray",
    linestyle="--",
    linewidth=0.8
)

plt.show()

The tick positions are set explicitly to land on each line of the triplet, which makes the coupling constant directly readable from the axis. The horizontal grid at 0, 0.4, and 0.8 intensity lets a reader verify the 1:2:1 ratio by eye. None of these choices affect the main axes, because the two axes objects are fully independent.

## 4. Inset as a secondary plot: GPC calibration inside a molecular weight distribution

An inset does not have to show a zoomed-in region of the main data. It can show a completely different but related plot. A common example in polymer characterization is a GPC (gel permeation chromatography) trace: the main plot shows the differential molecular weight distribution (dW/d log M vs. log M), and the inset shows the calibration curve that was used to convert elution volume to molecular weight. Placing the calibration inset inside the MWD figure keeps the methodology visible alongside the result.

In [ ]:
# Synthetic MWD: log-normal distribution in log10(M) space
log_mw_axis = np.linspace(3.5, 6.5, 600)       # log10(M) from ~3000 to ~3,000,000
mwd_center = 5.0                                 # peak at log M = 5 (Mn ~ 100 000 g/mol)
mwd_width = 0.35                                 # controls dispersity
mwd_curve = np.exp(-((log_mw_axis - mwd_center) ** 2) / (2 * mwd_width ** 2))

# Polystyrene calibration standards: elution volume (mL) and known log10(Mw)
cal_volume = np.array([14.2, 15.8, 17.1, 18.6, 20.0])   # elution volume in mL
cal_log_mw = np.array([5.90, 5.40, 4.88, 4.30, 3.75])   # log10(Mw) for each standard

# Fit a straight line through the calibration points
cal_coeffs = np.polyfit(cal_volume, cal_log_mw, deg=1)
cal_fit_x = np.linspace(13.5, 21.0, 200)
cal_fit_y = np.polyval(cal_coeffs, cal_fit_x)

In [ ]:
fig, ax_mwd = plt.subplots(figsize=(9, 5))

# Main plot: molecular weight distribution
ax_mwd.plot(log_mw_axis, mwd_curve, color="steelblue", linewidth=2.0)
ax_mwd.fill_between(log_mw_axis, mwd_curve, alpha=0.15, color="steelblue")
ax_mwd.set_xlabel("log$_{10}$(M) (g mol$^{-1}$)")
ax_mwd.set_ylabel("dW / d log M (normalized)")
ax_mwd.set_title("Molecular weight distribution with GPC calibration inset")
ax_mwd.set_xlim(3.5, 7.5)
ax_mwd.set_ylim(0, 1.15)

# Place calibration inset in the upper right, away from the MWD peak
ax_cal = inset_axes(
    ax_mwd,
    width="38%",
    height="48%",
    loc="upper right",
    bbox_to_anchor=(0.05, 0.05, 0.93, 0.88),
    bbox_transform=ax_mwd.transAxes
)

# Plot calibration data points and the linear fit
ax_cal.scatter(
    cal_volume, cal_log_mw,
    color="firebrick", s=40, zorder=3, label="PS standards"
)
ax_cal.plot(
    cal_fit_x, cal_fit_y,
    color="firebrick", linewidth=1.2, linestyle="--", label="Linear fit"
)

ax_cal.set_xlabel("Elution volume (mL)", fontsize=8)
ax_cal.set_ylabel("log$_{10}$(M)", fontsize=8)
ax_cal.set_title("GPC calibration", fontsize=9, pad=-10)
ax_cal.tick_params(labelsize=7)
ax_cal.legend(loc='upper right', fontsize=7, framealpha=0.8)

# Add a light border to visually separate the inset from the MWD
for spine in ax_cal.spines.values():
    spine.set_edgecolor("gray")
    spine.set_linewidth(0.8)

plt.show()

The calibration inset shows the five polystyrene standards and the linear fit in log M versus elution volume space. Placing it inside the MWD figure makes clear that the x-axis of the main plot was derived from this calibration, not measured directly. The inset occupies space that would otherwise be empty, since the MWD peak sits to the left of center.

## When to use an inset

An inset earns its place when the detail it shows cannot be read at the scale of the main figure, and when that detail is important enough that the reader should not have to hunt for it in a separate panel. It becomes clutter when the inset overlaps important features of the main plot, when the same information could be shown more cleanly by simply rescaling the main axes, or when the figure already carries enough visual complexity that one more layer of information impedes rather than aids understanding.